# DiscountMate — FAISS Index Builder (A100-optimised, Augmented)

Builds `faiss_index.bin` and `product_metadata.json` from MongoDB product data.
Architecture mirrors **`coles_reverse_image_search_Official_2`** — the key fix for
poor retrieval quality:

- **5 augmentations per product image** are indexed separately (normal, blur,
  saturate, flipped, centre_zoom) → 5× denser index coverage per product.
- **TTA (Test-Time Augmentation)** at query time: all 5 augments are searched,
  then scores are aggregated per product via *hybrid* strategy (0.7 × max + 0.3 × mean).
- **METRIC_INNER_PRODUCT** HNSW index: inner product on L2-normalised vectors = cosine similarity.

**Before running:**
1. Go to **Runtime → Change runtime type → A100 GPU**
2. Click the 🔑 icon → add secret `MONGO_URI`
3. Run all cells top-to-bottom

**Outputs** are saved to `My Drive/DiscountMate/faiss_output/`. Copy both files to
`experimental/ReverseImageSearch/`.

| Setting | Value | Why |
|---------|-------|-----|
| Augmentations | 5 per image | Matches Official notebook; each augment = separate index vector |
| Download batch | 128 imgs → 640 aug tensors | Fills GPU pipeline with minimal idle time |
| Inference batch | 640 | A100 40 GB handles ViT-B/14 × 640 in bfloat16 comfortably |
| Precision | bfloat16 | A100 native — 2× throughput vs float32 |
| torch.compile | yes | Fuses ViT ops, ~15–25% faster inference |
| TF32 | yes | Free ~10% on A100 matmuls |
| Download workers | 16 | Parallel HTTP threads overlap GPU work |
| HNSW efConstruction | 400 | Higher quality graph than Official's 200 |
| TTA strategy | hybrid | 0.7 × max + 0.3 × mean across 5 query augments |

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q pymongo faiss-cpu torch torchvision pillow opencv-python-headless requests tqdm

In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────────
import os
import sys
import json
import warnings
from concurrent.futures import ThreadPoolExecutor
from io import BytesIO
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import cv2
import faiss
import numpy as np
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from pymongo import MongoClient
from torchvision import transforms
from tqdm.notebook import tqdm

print("All imports OK")
print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Cell 3: Configuration ─────────────────────────────────────────────────────
try:
    from google.colab import userdata, drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    MONGO_URI  = userdata.get("MONGO_URI")
    OUTPUT_DIR = Path("/content/drive/MyDrive/DiscountMate/faiss_output")
else:
    # Local: set MONGO_URI in your environment or a .env file
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    MONGO_URI  = os.environ.get("MONGO_URI", "")
    OUTPUT_DIR = Path(".").resolve()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INDEX_PATH    = OUTPUT_DIR / "faiss_index.bin"
METADATA_PATH = OUTPUT_DIR / "product_metadata.json"

# ── MongoDB ────────────────────────────────────────────────────────────────────
MONGO_DB   = "DiscountMate_DB"
COLLECTION = "products"

# ── Augmentation (matches Official notebook architecture) ──────────────────────
# Each downloaded image generates len(AUGMENTS) vectors in the index.
# The same augments are applied to query images at search time (TTA).
AUGMENTS     = ["normal", "blur", "saturate", "flipped", "centre_zoom"]
TTA_STRATEGY = "hybrid"   # "max" | "hybrid" (0.7 × max + 0.3 × mean)

# ── A100 performance ───────────────────────────────────────────────────────────
# 128 downloads × 5 augments = 640 tensors per GPU pass — A100 40GB handles easily.
DOWNLOAD_BATCH   = 128    # original images per concurrent download chunk
INFER_BATCH_SIZE = 640    # max images per single bfloat16 forward pass
DOWNLOAD_WORKERS = 16     # parallel HTTP download threads
IMAGE_TIMEOUT    = 10     # seconds per download
USE_COMPILE      = True   # torch.compile — requires PyTorch >= 2.0

# TF32: free ~10% speedup on A100 matmuls with negligible precision loss
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

# ── FAISS ──────────────────────────────────────────────────────────────────────
VECTOR_DIM      = 768   # DINOv2 ViT-B/14 CLS token dimension
HNSW_M          = 32    # graph connectivity (matches Official notebook)
EF_CONSTRUCTION = 400   # build-time quality — higher than Official's 200
EF_SEARCH       = 64    # query-time search breadth

print(f"Device          : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Download batch  : {DOWNLOAD_BATCH}  →  {DOWNLOAD_BATCH * len(AUGMENTS)} aug tensors / GPU pass")
print(f"Infer batch size: {INFER_BATCH_SIZE}")
print(f"Augmentations   : {AUGMENTS}")
print(f"TTA strategy    : {TTA_STRATEGY}")
print(f"torch.compile   : {USE_COMPILE}")
print(f"Output dir      : {OUTPUT_DIR}")
print(f"Mongo URI       : {'set ✓' if MONGO_URI else '*** NOT SET — add MONGO_URI to Colab Secrets ***'}")

In [ ]:
# ── Cell 4: Image Augmentation (ported from Official notebook) ─────────────────
# Identical logic to coles_reverse_image_search_Official_2 Step 3.
# Operates on OpenCV BGR uint8 arrays; returns a new BGR array (input unchanged).

def augment_image(bgr: np.ndarray, augment: str) -> np.ndarray:
    """
    Apply one named augmentation to an OpenCV BGR image.

    normal      — identity copy
    blur        — Gaussian blur 15×15 (soft-focus / motion simulation)
    saturate    — HSV saturation ×1.5 (over-saturated packaging shot)
    flipped     — horizontal mirror
    centre_zoom — crop central 70%, resize back (zoom-in on label)
    """
    img = bgr.copy()
    if augment == "normal":
        return img
    elif augment == "blur":
        return cv2.GaussianBlur(img, (15, 15), 0)
    elif augment == "saturate":
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:, :, 1] = np.clip(hsv[:, :, 1] * 1.5, 0, 255)
        return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    elif augment == "flipped":
        return cv2.flip(img, 1)
    elif augment == "centre_zoom":
        h, w = img.shape[:2]
        cy, cx = h // 2, w // 2
        ch, cw = int(h * 0.7), int(w * 0.7)
        y1, x1 = cy - ch // 2, cx - cw // 2
        return cv2.resize(img[y1:y1 + ch, x1:x1 + cw], (w, h),
                          interpolation=cv2.INTER_LINEAR)
    else:
        raise ValueError(f"Unknown augmentation: '{augment}'")


# DINOv2 preprocessing — matches Official notebook and notebook_runtime.py
_PREPROCESS = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def _pil_to_aug_tensor(pil_img: Image.Image, aug: str) -> torch.Tensor:
    """PIL RGB → OpenCV BGR → augment → back to RGB PIL → preprocess tensor."""
    bgr     = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    aug_bgr = augment_image(bgr, aug)
    aug_rgb = cv2.cvtColor(aug_bgr, cv2.COLOR_BGR2RGB)
    return _PREPROCESS(Image.fromarray(aug_rgb))


print(f"Augmentation functions ready — {len(AUGMENTS)} augments: {AUGMENTS}")

In [ ]:
# ── Cell 5: Load DINOv2 ViT-B/14 ─────────────────────────────────────────────
# First run downloads ~330 MB from Torch Hub; subsequent runs use the cache.
# torch.compile fuses ViT ops for ~15–25% faster inference on A100.

def _patch_cached_dinov2_for_python39() -> bool:
    """Patch cached DINOv2 source for Python < 3.10 union-type syntax."""
    cache_root = (
        Path(torch.hub.get_dir()) / "facebookresearch_dinov2_main" / "dinov2" / "layers"
    )
    patched = False
    for target in [cache_root / "attention.py", cache_root / "block.py"]:
        if not target.exists():
            continue
        text    = target.read_text(encoding="utf-8")
        updated = text.replace("float | None", "Optional[float]")
        if updated != text and target.name == "attention.py" and "from typing import Optional" not in updated:
            updated = updated.replace("import os\n", "import os\nfrom typing import Optional\n", 1)
        if updated != text:
            target.write_text(updated, encoding="utf-8")
            patched = True
    return patched


def _load_dinov2_backbone() -> nn.Module:
    try:
        return torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")
    except TypeError as exc:
        if "unsupported operand type(s) for |" not in str(exc):
            raise
        if not _patch_cached_dinov2_for_python39():
            raise RuntimeError("Clear the DINOv2 Torch Hub cache and retry.") from exc
        for name in list(sys.modules):
            if name == "dinov2" or name.startswith("dinov2."):
                sys.modules.pop(name, None)
        return torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")


class DinoV2Extractor(nn.Module):
    """DINOv2 ViT-B/14 with L2-normalised CLS output — matches Official notebook."""
    def __init__(self) -> None:
        super().__init__()
        self.backbone = _load_dinov2_backbone()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.backbone(x), p=2, dim=1)


print("Loading DINOv2 ViT-B/14 (~330 MB on first run) ...")
model = DinoV2Extractor().to(DEVICE).eval()

if USE_COMPILE and hasattr(torch, "compile"):
    print("Compiling with torch.compile (~60 s first call, then fast) ...")
    model = torch.compile(model)

print("Model ready.")

In [ ]:
# ── Cell 6: Connect to MongoDB & fetch product list ───────────────────────────

client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=10_000)
client.admin.command("ping")
print("MongoDB connection OK")

db = client[MONGO_DB]
projection = {
    "_id": 1,
    "product_name": 1,
    "product_code": 1,
    "link_image": 1,
    "coles_price": 1,
    "woolworths_price": 1,
    "iga_price": 1,
}
query = {"link_image": {"$exists": True, "$nin": [None, ""]}}

print("Fetching products from MongoDB ...")
products: List[Dict] = list(db[COLLECTION].find(query, projection))
print(f"Found {len(products):,} products with image URLs")

In [ ]:
# ── Cell 7: Download + augment + extract embeddings ───────────────────────────
#
# Architecture (mirrors Official notebook build_pipeline):
#   1. Download DOWNLOAD_BATCH images concurrently via DOWNLOAD_WORKERS threads.
#   2. For each success, generate len(AUGMENTS)=5 augmented tensors.
#   3. Accumulate tensors into a buffer; flush to GPU every INFER_BATCH_SIZE.
#   4. bfloat16 autocast halves VRAM and doubles A100 throughput vs float32.
#   5. Metadata carries "augment" field so TTA query can group by product.
#
# Result: ~5× more index vectors than the single-vector approach,
# covering normal + degraded + flipped + saturated + zoomed views per product.

_HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; DiscountMate-IndexBuilder/1.0)"}


def _download(url: str) -> Optional[Image.Image]:
    try:
        r = requests.get(url, headers=_HEADERS, timeout=IMAGE_TIMEOUT)
        r.raise_for_status()
        return Image.open(BytesIO(r.content)).convert("RGB")
    except Exception:
        return None


def _price_str(val: Any) -> Optional[str]:
    if val is None:
        return None
    try:
        return f"{float(val):.2f}"
    except (TypeError, ValueError):
        return str(val)


def _build_meta(product: Dict, url: str, aug: str) -> Dict[str, Any]:
    return {
        "mongo_id":         str(product["_id"]),
        "title":            (product.get("product_name") or "").strip(),
        "image_url":        url,
        "product_code":     str(product.get("product_code") or ""),
        "coles_price":      _price_str(product.get("coles_price")),
        "woolworths_price": _price_str(product.get("woolworths_price")),
        "iga_price":        _price_str(product.get("iga_price")),
        "augment":          aug,   # which augmentation this vector was extracted from
    }


def _infer_batch(tensors: List[torch.Tensor]) -> np.ndarray:
    """Single forward pass (bfloat16 on CUDA). Returns (N, 768) float32."""
    batch = torch.stack(tensors).to(DEVICE)
    if DEVICE == "cuda":
        with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            vecs = model(batch)
    else:
        with torch.inference_mode():
            vecs = model(batch)
    return vecs.float().cpu().numpy().astype(np.float32)


# Build work list from MongoDB products with valid URLs
work_items: List[Tuple[str, Dict]] = []
n_no_url = 0
for p in products:
    url = (p.get("link_image") or "").strip()
    if url:
        work_items.append((url, p))
    else:
        n_no_url += 1

print(f"Products to process : {len(work_items):,}  |  No URL skipped: {n_no_url:,}")
print(f"Expected index size : ~{len(work_items) * len(AUGMENTS):,} vectors "
      f"({len(work_items):,} products × {len(AUGMENTS)} augments)")

all_embeddings : List[np.ndarray] = []
all_metadata   : List[Dict[str, Any]] = []
n_dl_fail  = 0
n_aug_fail = 0

# Rolling GPU batch buffers — flushed every INFER_BATCH_SIZE tensors
_tensor_buf : List[torch.Tensor]   = []
_meta_buf   : List[Dict[str, Any]] = []


def _flush_gpu() -> None:
    """Infer current buffer and append results to global lists."""
    if not _tensor_buf:
        return
    vecs = _infer_batch(_tensor_buf)
    all_embeddings.append(vecs)
    all_metadata.extend(_meta_buf)
    _tensor_buf.clear()
    _meta_buf.clear()


with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
    pbar = tqdm(total=len(work_items), desc="Download + Aug + Embed", unit="img")

    for chunk_start in range(0, len(work_items), DOWNLOAD_BATCH):
        chunk = work_items[chunk_start : chunk_start + DOWNLOAD_BATCH]

        # Submit all downloads for this chunk concurrently
        futures = {
            executor.submit(_download, url): (url, product)
            for url, product in chunk
        }

        for future, (url, product) in futures.items():
            pil_img = future.result()
            if pil_img is None:
                n_dl_fail += 1
                pbar.update(1)
                continue

            # Generate one tensor per augmentation — same as Official notebook
            for aug in AUGMENTS:
                try:
                    tensor = _pil_to_aug_tensor(pil_img, aug)
                except Exception as exc:
                    warnings.warn(f"[AUG FAIL] {url} [{aug}]: {exc}")
                    n_aug_fail += 1
                    continue
                _tensor_buf.append(tensor)
                _meta_buf.append(_build_meta(product, url, aug))

                if len(_tensor_buf) >= INFER_BATCH_SIZE:
                    _flush_gpu()

            pbar.update(1)

    pbar.close()

_flush_gpu()  # flush remaining tensors in the tail batch

print(f"\nEmbeddings extracted : {len(all_metadata):,}")
print(f"  Products × augments: {len(work_items) - n_dl_fail:,} × {len(AUGMENTS)} = "
      f"{(len(work_items) - n_dl_fail) * len(AUGMENTS):,} expected")
print(f"Download failures    : {n_dl_fail:,}")
print(f"Augment failures     : {n_aug_fail:,}")
if torch.cuda.is_available():
    print(f"Peak VRAM used       : {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

In [ ]:
# ── Cell 8: Build FAISS HNSW index & save ────────────────────────────────────
#
# METRIC_INNER_PRODUCT with L2-normalised vectors = cosine similarity.
# Matches Official notebook: IndexHNSWFlat(VECTOR_DIM, 32, faiss.METRIC_INNER_PRODUCT)
# efConstruction=400 gives higher quality graph than Official's 200 (build once, quality matters).

embeddings = np.vstack(all_embeddings)   # (N, 768)
print(f"Embedding matrix : {embeddings.shape}  dtype={embeddings.dtype}")

index = faiss.IndexHNSWFlat(VECTOR_DIM, HNSW_M, faiss.METRIC_INNER_PRODUCT)
index.hnsw.efConstruction = EF_CONSTRUCTION
index.hnsw.efSearch       = EF_SEARCH

print(f"Building HNSW index (efConstruction={EF_CONSTRUCTION}, METRIC_INNER_PRODUCT) ...")
index.add(embeddings)
print(f"FAISS index built : {index.ntotal:,} vectors")

faiss.write_index(index, str(INDEX_PATH))
print(f"Saved → {INDEX_PATH}")

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(all_metadata, f, ensure_ascii=False)
print(f"Saved → {METADATA_PATH}")

In [ ]:
# ── Cell 9: Verify output & TTA query function ────────────────────────────────
#
# Verification: reload both files and run a sanity-check search.
# TTA query: mirrors Official notebook query_product — 5 augments batched
# into one forward(), scores aggregated by mongo_id via TTA_STRATEGY.

index_mb = INDEX_PATH.stat().st_size / 1_048_576
meta_mb  = METADATA_PATH.stat().st_size / 1_048_576

chk_index = faiss.read_index(str(INDEX_PATH))
with open(METADATA_PATH, encoding="utf-8") as f:
    chk_meta = json.load(f)

assert chk_index.ntotal == len(chk_meta), (
    f"Mismatch: {chk_index.ntotal} vectors vs {len(chk_meta)} metadata entries"
)
dummy = np.zeros((1, VECTOR_DIM), dtype=np.float32)
_, test_ids = chk_index.search(dummy, 1)
assert test_ids[0][0] != -1, "FAISS returned -1 — index may be empty"

unique_products = len({m["mongo_id"] for m in chk_meta})

print("=" * 54)
print(" Index build complete")
print("=" * 54)
print(f"  faiss_index.bin       : {index_mb:.1f} MB  ({chk_index.ntotal:,} vectors)")
print(f"  product_metadata.json : {meta_mb:.1f} MB  ({len(chk_meta):,} entries)")
print(f"  Unique products       : {unique_products:,}  ({len(AUGMENTS)} augments each)")
print(f"  Download failures     : {n_dl_fail:,}")
print(f"  Augment failures      : {n_aug_fail:,}")
print()


# ── TTA Query Function — mirrors Official notebook query_product ──────────────

def query_product(
    image_url_or_path: str,
    faiss_index,
    metadata: list,
    top_k: int = 5,
) -> list:
    """
    Reverse-image search with Test-Time Augmentation.

    Applies all 5 augments to the query, searches FAISS for each, then
    aggregates scores per product (by mongo_id) using TTA_STRATEGY:
      'max'    → best score across augments
      'hybrid' → 0.7 × max + 0.3 × mean  (rewards multi-augment matches)

    Args:
        image_url_or_path: HTTP URL or local file path of the query image.
        faiss_index: loaded FAISS index (METRIC_INNER_PRODUCT; higher = better).
        metadata: parallel list of metadata dicts from product_metadata.json.
        top_k: number of unique products to return.

    Returns:
        Ranked list of dicts: rank, mongo_id, title, product_code, prices,
        augment, image_url, similarity_score.
    """
    # Load query image
    if image_url_or_path.startswith("http"):
        r = requests.get(image_url_or_path, headers=_HEADERS, timeout=IMAGE_TIMEOUT)
        r.raise_for_status()
        pil_img = Image.open(BytesIO(r.content)).convert("RGB")
    else:
        pil_img = Image.open(image_url_or_path).convert("RGB")

    # One batched forward pass for all 5 query augments
    aug_tensors = [_pil_to_aug_tensor(pil_img, aug) for aug in AUGMENTS]
    aug_batch   = torch.stack(aug_tensors).to(DEVICE)
    if DEVICE == "cuda":
        with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            aug_vecs = model(aug_batch)
    else:
        with torch.inference_mode():
            aug_vecs = model(aug_batch)
    aug_vecs_np = aug_vecs.float().cpu().numpy().astype(np.float32)

    fetch_k = top_k * 15   # extra candidates to keep top_k unique after dedup

    # Aggregate per-product across all augment searches
    # mongo_id → {max, sum, count, best_meta}
    product_agg: Dict = {}
    for vec in aug_vecs_np:
        q = np.ascontiguousarray(vec.reshape(1, -1), dtype=np.float32)
        scores, idxs = faiss_index.search(q, fetch_k)
        for fidx, score in zip(idxs[0], scores[0]):
            if fidx == -1:
                continue
            m   = metadata[fidx]
            pid = m["mongo_id"]
            s   = float(score)
            entry = product_agg.get(pid)
            if entry is None:
                product_agg[pid] = {"max": s, "sum": s, "count": 1, "meta": m}
            else:
                if s > entry["max"]:
                    entry["max"]  = s
                    entry["meta"] = m   # track the augment that scored best
                entry["sum"]   += s
                entry["count"] += 1

    # Reduce to a single score per product
    scored = []
    for pid, agg in product_agg.items():
        if TTA_STRATEGY == "hybrid":
            final = 0.7 * agg["max"] + 0.3 * (agg["sum"] / agg["count"])
        else:
            final = agg["max"]
        scored.append((final, agg["meta"]))

    scored.sort(key=lambda x: x[0], reverse=True)

    return [
        {
            "rank":             rank,
            "mongo_id":         m["mongo_id"],
            "title":            m["title"],
            "product_code":     m.get("product_code", ""),
            "coles_price":      m.get("coles_price"),
            "woolworths_price": m.get("woolworths_price"),
            "iga_price":        m.get("iga_price"),
            "augment":          m["augment"],
            "image_url":        m["image_url"],
            "similarity_score": score,
        }
        for rank, (score, m) in enumerate(scored[:top_k], 1)
    ]


# ── Smoke test — query the first indexed product against itself ────────────────
print("Sample metadata entry [0]:")
print(json.dumps(chk_meta[0], indent=2, ensure_ascii=False))
print()

first_url = chk_meta[0]["image_url"]
print(f"Running smoke-test query on: {first_url}")
try:
    results = query_product(first_url, chk_index, chk_meta, top_k=5)
    print(f"\nTop-5 results (TTA strategy: {TTA_STRATEGY}):")
    for r in results:
        print(f"  #{r['rank']}  {r['similarity_score']:.4f}  "
              f"[{r['mongo_id']}]  {r['title'][:60]}  (aug={r['augment']})")
    first_id = chk_meta[0]["mongo_id"]
    hit_rank = next((r["rank"] for r in results if r["mongo_id"] == first_id), None)
    print()
    if hit_rank == 1:
        print("Self-retrieval check: PASS (correct product ranked #1)")
    elif hit_rank:
        print(f"Self-retrieval check: OK (correct product ranked #{hit_rank})")
    else:
        print("Self-retrieval check: product not in top-5 — inspect index quality")
except Exception as exc:
    print(f"Smoke-test skipped: {exc}")

print()
print("Next steps:")
print("  1. Download faiss_index.bin + product_metadata.json from:")
print(f"     My Drive → DiscountMate/faiss_output/")
print("  2. Place in: experimental/ReverseImageSearch/")
print("  3. Restart the API:")
print("       uvicorn api:app --host 127.0.0.1 --port 8001")
print()
print("Note: the index uses METRIC_INNER_PRODUCT (higher score = better match).")
print("Update api.py if it previously filtered/sorted on L2 distances.")